# 09 — Panel App

## The same pipeline, served as an interactive web app
---

**Format:** Narrated demo — pre-run outputs. Runs in under 7 minutes.

This module is a standalone addendum. It answers one question: **once you have a validated, production-grade pipeline from Modules 01–07, how do you turn it into an interactive app a non-Python user can drive?**

```
Panel + Material UI    Python web app served by a Bokeh server
                       Reactive widgets bound to param.Parameter values
                       hvplot/HoloViews for the lightcurve + score plots
                       panel-material-ui for polished MUI components
                       panel serve app.py → localhost:5006
                       Stewarded by Anaconda (HoloViz ecosystem).
```

**The data:** Five real exoplanet systems with light curves synthesised from published TESS parameters. Same `PHASE / LC_DETREND / MODEL_INIT` schema as Module 01.

**The pipeline:** Same IsolationForest anomaly detection and validation stats as `ingestion.py` from Module 01.

**What changes:** only the delivery layer. The pipeline is portable because its outputs are typed and structured.

## The five targets

The data is synthesised from published orbital parameters using a limb-darkened transit model + realistic TESS photometric noise — no CSV file needed.

| Target | Type | Period | Transit depth | What makes it interesting |
|---|---|---|---|---|
| **WASP-18 b** | Hot Jupiter | 0.94 days | 1.01% | Shortest period, clearest signal — same system as Module 01 |
| **WASP-12 b** | Hot Jupiter | 1.09 days | 1.43% | Tidally disrupting, deeper transit, higher noise |
| **HD 209458 b** | Hot Jupiter | 3.52 days | 1.47% | First confirmed transiting exoplanet (1999), lowest noise |
| **TRAPPIST-1 e** | Rocky — habitable zone | 6.10 days | 0.35% | Deliberately hard — shallow signal, M dwarf host noise |
| **Kepler-7 b** | Inflated Hot Jupiter | 4.89 days | 0.83% | Styrofoam density, long transit duration, interesting residuals |

# Panel + Material UI

Panel is part of [HoloViz](https://holoviz.org), the visualisation ecosystem stewarded by Anaconda. It sits on top of Bokeh and adds a reactive, `param`-based widget layer — the same `param` you use for class-level config becomes the binding glue for the UI.

`panel-material-ui` brings Material Design components (Select, Button, Paper, Tabs, TextAreaInput) on top of Panel — the polish in the sidebar and tab bar comes from there rather than vanilla Bokeh widgets.

The entire app is `app.py` — a single file holding the pipeline functions and the `LightcurveExplorer` class that wires them to widgets and plots.

## Running it

```
pip install panel panel-material-ui hvplot scikit-learn pandas numpy
panel serve app.py --show 
```

That's it. No CSV to copy. Select a target from the sidebar, press Run Analysis.

What runs where:
  Bokeh server     — Python process, hosts the pipeline + widget state
  Browser tab      — receives plot/widget updates over a websocket
  param.Parameter  — the binding layer between widgets and your model

Dependencies (all conda-forge / PyPI):
  panel               — reactive web app framework on top of Bokeh
  panel-material-ui   — Material UI components for Panel
  hvplot              — pandas .hvplot accessor → HoloViews
  holoviews           — declarative plotting (transitive)
  scikit-learn        — IsolationForest
  pandas / numpy      — pipeline core

No build step. No install for the end user beyond pointing a browser at the URL.""")


## What the app does

1. **Step 1 — Choose a target** from the sidebar `Select`. The info card at the top updates reactively (`@param.depends('target_name', watch=True)`) with the system's physical properties.
2. **Step 2 — Press Run Analysis.** The pipeline executes in the server process and streams updates to the browser:
   - Synthesise light curve from published parameters (Mandel-Agol limb-darkened transit + TESS noise)
   - Validate schema, nulls, flux std, duplicate phases
   - IsolationForest anomaly detection (`contamination=0.05`, same params as Module 01)
   - Each step writes to the pipeline log in real time
3. **Step 3 — Four tabs populate** in a single `Paper` card:
   - **Pipeline log** — `TextAreaInput` showing the four-stage trace (synthesise → validate → detect → render)
   - **Validation Report** — Markdown pane with the same fields as Module 01's `ValidationReport`: rows, nulls, duplicates, flux std, phase/flux ranges
   - **Anomalous Points** — `Tabulator` of the top 50 anomalies, sortable, with native pagination
   - **Lightcurve + Anomaly Scores** — phase-folded flux (anomalies in crimson) over the score panel with a dashed threshold line
4. **Step 4 — Reselect a target** at any time. The info card refreshes instantly; the reports stay hidden until the next Run.

## The Panel pattern

Everything reactive in Panel is a `param.Parameter` or a Panel widget bound to one. There is no global state and no manual callback wiring beyond the Run button.

```python
import panel as pn
import panel_material_ui as pmui
import param

pn.extension("tabulator")

class LightcurveExplorer(pn.viewable.Viewer):
    target_name = param.Selector(default=TARGET_OPTIONS[0], objects=TARGET_OPTIONS)

    def __init__(self, **params):
        super().__init__(**params)

        # Widgets bound to the param — picking a value updates target_name
        self.target_widget = pmui.Select.from_param(self.param.target_name, label="Target system")
        self.run_button    = pmui.Button(name="Run Analysis", color="primary", icon="analytics")
        self.run_button.on_click(self._on_run)

        # Output panes
        self.info_pane = pn.pane.Markdown("", sizing_mode="stretch_width")
        self.log_widget = pmui.TextAreaInput(disabled=True, height=250)
        self.stats_md  = pn.pane.Markdown("", sizing_mode="stretch_width")
        self.table     = pn.widgets.Tabulator(pagination="local", page_size=12, height=320)
        self.plot_pane = pn.pane.HoloViews(sizing_mode="stretch_width")

        self.reports = pmui.Paper(
            pmui.Tabs(
                ("Pipeline log",                self.log_widget),
                ("Validation Report",           self.stats_md),
                ("Anomalous Points",            self.table),
                ("Lightcurve + Anomaly Scores", self.plot_pane),
            ),
            visible=False,
        )

    @param.depends("target_name", watch=True)
    def _refresh_info(self):
        # Runs automatically whenever target_name changes — no callback registration
        self.info_pane.object = render_target_md(self.target_name)
        self.reports.visible = False

    def _on_run(self, _event=None):
        key = TARGET_NAME_TO_KEY[self.target_name]
        phase, lc, model = _synthesise(key)         # same pipeline as ingestion.py
        report           = _validate(phase, lc, model)
        result           = _detect_anomalies(phase, lc, model)
        self._populate_stats(TARGETS[key], report, result)
        self._populate_table(result)
        self._build_plot(phase, lc, model, result)
        self.reports.visible = True

    def __panel__(self):
        return pmui.Page(
            title="Exoplanet Lightcurve Explorer",
            sidebar=[self.target_widget, self.run_button],
            main=[self.info_card, self.reports],
        )

LightcurveExplorer().servable()
```

Key points:
- `param.Selector` on the class is the source of truth — the widget is bound to it via `Select.from_param`
- `@param.depends(..., watch=True)` registers the reactive callback declaratively — no `.observe()` plumbing
- `pmui.Page` lays out sidebar + main area with a Material theme; `pmui.Paper` and `pmui.Tabs` give the card-and-tab structure
- `.servable()` marks the object as the document root so `panel serve` knows what to render

## hvplot for the plots

Panel composes well with HoloViews/hvplot — the same plotting layer the rest of the HoloViz stack uses. The lightcurve panel is two `hvplot.points` overlays plus an `HLine` for the anomaly threshold:

```python
import hvplot.pandas  # registers .hvplot on DataFrames
import holoviews as hv

results = pd.DataFrame({
    "phase": phase, "lc_detrend": lc, "model_init": model,
    "anomaly_score": result["scores"], "is_anomaly": result["is_anomaly"],
})
normal    = results[~results["is_anomaly"]]
anomalous = results[ results["is_anomaly"]]

normal_points    = normal.hvplot.points(x="phase", y="lc_detrend", alpha=0.2, label="Normal")
anomalous_points = anomalous.hvplot.points(x="phase", y="lc_detrend", color="crimson", label="Anomaly")
score_points     = results.hvplot.points(x="phase", y="anomaly_score", color="darkorange")

threshold = float(anomalous["anomaly_score"].min())
hline     = hv.HLine(threshold).opts(color="crimson", line_dash="dashed")

self.plot_pane.object = (normal_points * anomalous_points + score_points * hline).cols(1)
```

The `*` operator overlays plots, `+` puts them in a layout, `.cols(1)` stacks them vertically. The `HoloViews` pane in Panel renders the resulting object directly — no manual figure management.

## How the pieces fit

```
param.Selector ──► panel_material_ui.Select ──► self.target_name
                                                       │
                              ┌────────────────────────┤
                              │                        │
                              ▼                        ▼
                  @param.depends(watch=True)    (user clicks Run)
                  → _refresh_info()              → _on_run()
                       │                              │
                       ▼                              ▼
                   info_pane                  _synthesise(key)
                                                      │
                                                      ▼
                                          _validate(phase, lc, model)
                                              │
                                              ├──► stats_md      (Markdown pane)
                                              │
                                              ▼
                                       _detect_anomalies(...)
                                              │
                                              ├──► table         (Tabulator)
                                              ├──► plot_pane     (HoloViews)
                                              └──► log_widget    (TextAreaInput, live)
                                                      │
                                                      ▼
                                              reports.visible = True
```

Every reactive piece is a `param.Parameter` or a Panel widget bound to one. The arrows are either `param.depends` reactivity (free) or direct attribute assignment in the Run handler (explicit). No global state.

---
## What Panel adds over the bare pipeline

| | `ingestion.py` (Module 01) | `app.py` (this module) |
|---|---|---|
| **Delivery** | CLI / library import | Web app served on a port |
| **Audience** | Python users | Anyone with a browser link |
| **Target selection** | Function argument | `pmui.Select` in the sidebar |
| **Validation report** | `ValidationReport` dataclass to stdout | Markdown pane in a tab |
| **Anomaly table** | DataFrame in REPL | `Tabulator` with sort + pagination |
| **Plots** | `matplotlib.pyplot.show()` | `hvplot` → interactive HoloViews |
| **Pipeline log** | stdout | Live-updating `TextAreaInput` tab |
| **Reactivity** | None — re-run the script | `@param.depends(watch=True)` |
| **Deployment** | `python -m ingestion` | `panel serve app.py` (or via Bokeh server, FastAPI, etc.) |
| **Best for** | Batch jobs, notebooks, CI | Stakeholder demos, internal tools, dashboards |
| **Lines of app code** | ~400 (pipeline only) | 290 (pipeline + UI) |

---

The pipeline is portable because its outputs are structured. The same `ValidationReport` fields — nulls, flux_std, phase_range, anomaly count, transit depth — that printed to stdout in Module 01 now populate Markdown panes and Tabulator rows in the Panel app. That structure came from `ingestion.py` and hasn't changed.

Panel is the delivery layer. The science stayed put.